# LLM Asset Classification Verification 

In [32]:
import pandas as pd
import os
from sentence_transformers import SentenceTransformer

data = pd.read_csv('test2.csv')
data = data

In [33]:
import requests
from huggingface_hub import configure_http_backend

def backend_factory() -> requests.Session:
    session = requests.Session()
    session.verify = False
    return session

configure_http_backend(backend_factory=backend_factory)

In [34]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

# The sentences to encode
sentences = data["Description"].dropna()  # Remove NaNs
sentences = sentences[sentences.apply(lambda x: isinstance(x, str))]  # Keep only strings
sentences = sentences.tolist()

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 384]

# 3. Calculate the embedding similarities
#similarities = model.similarity(embeddings, embeddings)
#print(similarities)
# tensor([[1.0000, 0.6660, 0.1046],
#         [0.6660, 1.0000, 0.1411],
#         [0.1046, 0.1411, 1.0000]])

C:\Users\medney\AppData\Roaming\Python\Python312\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\medney\AppData\Roaming\Python\Python312\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\medney\AppData\Roaming\Python\Python312\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C

(109605, 384)


In [35]:
import pandas as pd
import numpy as np

from sklearn.experimental import enable_hist_gradient_boosting  # Required to use HistGradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import classification_report
from tqdm import tqdm  # Progress bar
from joblib import Parallel, delayed
from sklearn.base import clone  # Cloning the estimator

# Custom OneVsRestClassifier that parallelizes training with a working progress bar.
class OneVsRestClassifierParallel:
    def __init__(self, estimator, n_jobs=-1):
        self.estimator = estimator  # Store the estimator
        self.n_jobs = n_jobs
        self.classifiers_ = []  # List to hold individual classifiers

    def fit(self, X, y):
        n_classes = y.shape[1]  # Number of classifiers to train
        self.classifiers_ = [None] * n_classes  # Placeholder

        for i in tqdm(range(n_classes), desc="Training Progress"):
            clf = clone(self.estimator)  # Clone the estimator to avoid overwriting
            clf.fit(X, y[:, i])
            self.classifiers_[i] = clf  # Store the trained classifier

        return self

    def predict(self, X):
        predictions = np.array([clf.predict(X) for clf in self.classifiers_]).T
        return predictions
    
    def predict_proba(self, X):
        """Aggregates probability predictions from all classifiers."""
        prob_predictions = np.array([clf.predict_proba(X)[:, 1] for clf in self.classifiers_]).T
        return prob_predictions


# ----------------------------
# 1. Extract Features and Target
# ----------------------------
X = np.array(embeddings)  # Ensure embeddings is defined
y_raw = data['First_Match'].astype(str).values.flatten()  # Convert labels to strings

# ----------------------------
# 2. Compute Class Distribution
# ----------------------------
class_counts = pd.Series(y_raw).value_counts()
majority_class_size = class_counts.max()  # Largest class size

# Define sampling strategy
undersample_size = int(0.25 * majority_class_size)  # Reduce the majority class size by 75%
sampling_strategy_undersample = {class_label: min(count, undersample_size) for class_label, count in class_counts.items()}
sampling_strategy_oversample = {cls: max(int(0.1 * undersample_size), count) for cls, count in class_counts.items()}

# ----------------------------
# 3. Apply Hybrid Sampling (Under + Over Sampling)
# ----------------------------
# Step 1: Reduce the majority class
rus = RandomUnderSampler(sampling_strategy=sampling_strategy_undersample, random_state=42)
X_resampled, y_resampled = rus.fit_resample(X, y_raw)

# Step 2: Increase minority classes
ros = RandomOverSampler(sampling_strategy=sampling_strategy_oversample, random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_resampled, y_resampled)

# ----------------------------
# 4. One-Hot Encode the Target Labels
# ----------------------------
y_resampled = y_resampled.reshape(-1, 1)
onehot_encoder = OneHotEncoder(sparse_output=False)
y_encoded = onehot_encoder.fit_transform(y_resampled)

# ----------------------------
# 5. Train/Test Data Split
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_encoded, test_size=0.20, random_state=42)

# ----------------------------
# 6. Train the Model with Parallelized OneVsRestClassifier
# ----------------------------
clf = OneVsRestClassifierParallel(
    estimator=HistGradientBoostingClassifier(
        max_iter=300, 
        learning_rate=0.1, 
        max_depth=4, 
        max_leaf_nodes=31, 
        early_stopping=True, 
        validation_fraction=0.1, 
        n_iter_no_change=10, 
        random_state=0
    ),
    n_jobs=-1
)

clf.fit(X_train, y_train)

# ----------------------------
# 7. Evaluate the Model
# ----------------------------
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))


Training Progress: 100%|██████████| 296/296 [2:28:57<00:00, 30.19s/it]  


              precision    recall  f1-score   support

           0       1.00      1.00      1.00       216
           1       0.99      1.00      1.00       198
           2       0.99      1.00      1.00       194
           3       0.32      0.58      0.42       181
           4       0.63      0.51      0.56       251
           5       0.84      0.88      0.86       191
           6       0.92      0.89      0.91       198
           7       0.98      1.00      0.99       175
           8       0.52      0.51      0.51       183
           9       0.85      0.89      0.87       184
          10       0.79      0.67      0.72       206
          11       0.99      1.00      1.00       199
          12       1.00      1.00      1.00       200
          13       0.87      0.90      0.89       198
          14       1.00      0.15      0.26       172
          15       0.92      1.00      0.96       167
          16       0.67      0.72      0.70       204
          17       0.98    

C:\Users\medney\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [36]:
with open('model2.pkl', 'wb') as file:
    pickle.dump(clf, file)

with open('embedding.pkl', 'wb') as file:
    pickle.dump(model, file)

with open('one_hot.pkl', 'wb') as file:
    pickle.dump(onehot_encoder, file)

#file = open('model2.pkl', 'rb')
#clf = pickle.load(file)
#file.close()

In [37]:
# Define the custom class
class OneVsRestClassifierParallel:
    def __init__(self, estimator, n_jobs=-1):
        self.estimator = estimator
        self.n_jobs = n_jobs
        self.classifiers_ = []

    def fit(self, X, y):
        import numpy as np
        from sklearn.base import clone
        from tqdm import tqdm
        n_classes = y.shape[1]
        self.classifiers_ = [None] * n_classes
        for i in tqdm(range(n_classes), desc="Training Progress"):
            clf = clone(self.estimator)
            clf.fit(X, y[:, i])
            self.classifiers_[i] = clf
        return self

    def predict(self, X):
        import numpy as np
        predictions = np.array([clf.predict(X) for clf in self.classifiers_]).T
        return predictions

    def predict_proba(self, X):
        import numpy as np
        prob_predictions = np.array([clf.predict_proba(X)[:, 1] for clf in self.classifiers_]).T
        return prob_predictions

# Now load your model
import pickle

with open('model2.pkl', 'rb') as file:
    clf = pickle.load(file)


In [39]:
import numpy as np

def predict_top_3(model, input_string, onehot_encoder, text_to_embedding_func):
    """
    Predicts the top 3 most likely classes for a given input string.

    Parameters:
    - model: Trained OneVsRestClassifierParallel model
    - input_string: The text input to classify
    - onehot_encoder: OneHotEncoder used for training (to decode predictions)
    - text_to_embedding_func: Function to convert text to embeddings (ensure it matches training embeddings)

    Returns:
    - List of tuples (class_label, probability)
    """
    # Convert input string to embeddings
    X_input = text_to_embedding_func(input_string).reshape(1, -1)  # Ensure correct shape

    # Get probability predictions
    y_proba = model.predict_proba(X_input)

    # Get top 3 class indices
    top_3_indices = np.argsort(y_proba[0])[-3:][::-1]  # Get 3 highest probabilities

    # Convert class indices back to labels
    class_labels = onehot_encoder.categories_[0]  # Get original class labels
    top_3_labels = class_labels[top_3_indices]

    # Get top 3 probabilities
    top_3_probs = y_proba[0][top_3_indices]

    # Return as a list of tuples (label, probability)
    return list(zip(top_3_labels, top_3_probs))

# Example usage
input_text = "desktop computer, scada interface"
top_3_predictions = predict_top_3(clf, input_text, onehot_encoder, model.encode)

# Print results
for rank, (label, prob) in enumerate(top_3_predictions, start=1):
    print(f"{rank}. {label}: {prob:.4f}")


1. http://www.toronto.ca/TWONTO#00338: 1.0000
2. http://www.toronto.ca/TWONTO#00653: 1.0000
3. http://www.toronto.ca/TWONTO#00111: 0.2430


In [42]:
data[data['First_Match'] == 'http://www.toronto.ca/TWONTO#00368']

,Unnamed: 0,Entity_number,First_Match,Description
2929,5273,COL-PCU-FQ-0001,http://www.toronto.ca/TWONTO#00368,"Meter, Flow, Totalizer, City Water, Cumber PS"
3387,6826,COL-PKP-FQ-0001,http://www.toronto.ca/TWONTO#00368,"Meter, Flow"
3455,7127,COL-PLB-FQ-0001,http://www.toronto.ca/TWONTO#00368,"Meter, Flow"
3714,8011,COL-PMM-FQ-0001,http://www.toronto.ca/TWONTO#00368,"Meter, Flow"
3728,8059,COL-PMM-INLET FLOW,http://www.toronto.ca/TWONTO#00368,Inlet Flow Meter
3933,8797,COL-PNT-FQ-0001,http://www.toronto.ca/TWONTO#00368,"Meter, Flow City Water"
5007,11817,COL-PSW-FQ-0001,http://www.toronto.ca/TWONTO#00368,"Meter, Flow"
5177,12497,COL-PWN-FQ-0001,http://www.toronto.ca/TWONTO#00368,"Meter, Flow"
7156,17547,FCL-BFP-TESTKIT-0001,http://www.toronto.ca/TWONTO#00368,"Test Kit, Backflow Preventer Sr.No. 457813"
8713,20888,COL-PBG-FQ-0001,http://www.toronto.ca/TWONTO#00368,"Meter, Flow"
